# 05 — Capacity matching by bisection

This notebook confirms how the benchmark equalises model capacity. A reference architecture defines a target parameter budget; every other architecture is rescaled by bisection on its width scale until its parameter count lands within tolerance of that target. We drive the real matcher and plot its convergence history.

Modules exercised: `pipelines/benchmark_pipeline/sizing.py` (`SizeMatcher`, `SizeMatcher.match`, `SizeMatchResult`) and `configuration/benchmark_config.py` (`SizeMatchConfig`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## Configuring a fast, self-contained matcher

We use the real `SizeMatcher` but tighten its iteration budget and shrink the patch size so the demonstration runs quickly on CPU. The matcher writes to a `Logger` pointed at a temporary directory; nothing touches the data mount.

In [ ]:
import tempfile
from pathlib import Path

from configuration.benchmark_config import BenchmarkConfig
from pipelines.benchmark_pipeline.sizing import SizeMatcher
from tools.logger import Logger

config = BenchmarkConfig()
config.training.patch_size       = (32, 32)
config.size_match.max_iterations = 24
config.size_match.tolerance      = 0.01

log_dir = Path(tempfile.mkdtemp())
logger  = Logger(log_dir=str(log_dir), name="size_match_demo")
matcher = SizeMatcher(config=config, logger=logger)

print("tolerance     :", config.size_match.tolerance)
print("max_iterations:", config.size_match.max_iterations)
print("reference     :", config.size_match.reference_model)

## Choosing a target budget

To keep the demonstration light we set an explicit modest target rather than counting the full-size reference. The matching procedure is identical regardless of where the target comes from.

In [ ]:
TARGET = 300_000
print(f"target budget: {TARGET:,} parameters")

## Matching one architecture

We match a lightweight architecture to the target and inspect the returned `SizeMatchResult`: the chosen scale, the achieved parameter count, the deviation from target, the number of iterations, and the full bisection history.

In [ ]:
result = matcher.match("segformer", target=TARGET)

print(f"model         : {result.model}")
print(f"chosen scale  : {result.scale:.4f}")
print(f"parameters    : {result.parameters:,}")
print(f"target        : {result.target:,}")
print(f"deviation     : {result.deviation_pct:+.3f} %")
print(f"iterations    : {result.iterations}")
print(f"overrides     : {result.overrides}")

## Convergence history

The matcher records every probe. The upper panel shows the probed parameter count approaching the target band; the lower panel shows the absolute deviation shrinking toward the tolerance line. Together they make the bisection visible.

In [ ]:
history     = result.history
iters       = [h["iteration"] for h in history]
params      = [h["parameters"] for h in history]
deviations  = [abs(h["deviation_pct"]) for h in history]
tol_pct     = 100.0 * config.size_match.tolerance

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(7.5, 7), sharex=True)

ax_top.plot(iters, np.array(params) / 1e3, marker="o", color="#3b6ea5")
ax_top.axhline(TARGET / 1e3, color="black", ls="--", label="target")
ax_top.set_ylabel("probed parameters (thousands)")
ax_top.set_title(f"Bisection toward {TARGET:,} parameters ({result.model})")
ax_top.legend()

ax_bot.semilogy(iters, deviations, marker="s", color="#a5533b")
ax_bot.axhline(tol_pct, color="green", ls="--", label=f"tolerance {tol_pct:.1f} %")
ax_bot.set_xlabel("iteration")
ax_bot.set_ylabel("|deviation| (%)")
ax_bot.legend()

fig.tight_layout()
plt.show()

logger.close()

## Expected visual outcome

The result reports a parameter count close to the 300,000 target with a small signed deviation. The upper curve settles onto the dashed target line and the lower curve descends toward (and ideally below) the tolerance line, demonstrating that the bisection converges.